<a href="https://colab.research.google.com/github/Rundrayer/Evaluacion_Practica_Grupal_IDS221/blob/main/modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install scikit-learn

In [1]:
#IMPORTACION DE LIBRERIAS Y CARGA DE DATOS

# IMPORTAR LIBRERÍAS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from Configuracion import DOC_ANALISIS, DOC_RESULTADOS, DT_DATASET, DT_PROCESADOS, DT_VISTA

# Scikit-learn: modelos y métricas
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configurar estilo de gráficos
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# CARGAR Y ORDENAR DATOS
ruta_df = DT_DATASET /  'demanda_diaria.csv'
df = pd.read_csv(ruta_df)

df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values("fecha").reset_index(drop=True)

print("INFORMACIÓN DEL DATASET:")
print("=" * 50)
df.info()

print("\nVALORES NULOS POR COLUMNA:")
print(df.isnull().sum())

print(f"\nRANGO TEMPORAL:")
print(f"   Desde: {df['fecha'].min()}")
print(f"   Hasta: {df['fecha'].max()}")

INFORMACIÓN DEL DATASET:
<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   fecha              180 non-null    datetime64[us]
 1   demanda_kwh        180 non-null    float64       
 2   evento_programado  180 non-null    int64         
dtypes: datetime64[us](1), float64(1), int64(1)
memory usage: 4.3 KB

VALORES NULOS POR COLUMNA:
fecha                0
demanda_kwh          0
evento_programado    0
dtype: int64

RANGO TEMPORAL:
   Desde: 2026-01-01 00:00:00
   Hasta: 2026-06-29 00:00:00


In [2]:
# CREAR VARIABLES DE CALENDARIO
df["dia_semana"] = df["fecha"].dt.dayofweek  # 0=lunes, 6=domingo
df["mes"] = df["fecha"].dt.month

print("VARIABLES DE CALENDARIO CREADAS:")
print("=" * 60)
print(df[["fecha", "dia_semana", "mes"]].head(5).to_string())

VARIABLES DE CALENDARIO CREADAS:
       fecha  dia_semana  mes
0 2026-01-01           3    1
1 2026-01-02           4    1
2 2026-01-03           5    1
3 2026-01-04           6    1
4 2026-01-05           0    1


In [3]:
# CREAR VARIABLES DE REZAGO (LAG)
# shift(1) mueve los valores una posición hacia abajo
# Es decir: lag_1 en la fila 10 = demanda_kwh de la fila 9

df["lag_1"] = df["demanda_kwh"].shift(1)   # Demanda de ayer
df["lag_7"] = df["demanda_kwh"].shift(7)   # Demanda de hace 7 días

print("VARIABLES DE REZAGO CREADAS:")
print("=" * 70)
print(df[["fecha", "demanda_kwh", "lag_1", "lag_7"]].head(10).to_string())

print("\nOBSERVEN: Las primeras filas tienen NaN en lag_1 y lag_7")
print("   Porque no hay datos anteriores disponibles")

VARIABLES DE REZAGO CREADAS:
       fecha  demanda_kwh     lag_1     lag_7
0 2026-01-01     17856.31       NaN       NaN
1 2026-01-02     17379.10  17856.31       NaN
2 2026-01-03     15720.70  17379.10       NaN
3 2026-01-04     15694.05  15720.70       NaN
4 2026-01-05     16505.77  15694.05       NaN
5 2026-01-06     16937.36  16505.77       NaN
6 2026-01-07     17456.69  16937.36       NaN
7 2026-01-08     16568.10  17456.69  17856.31
8 2026-01-09     15495.83  16568.10  17379.10
9 2026-01-10     15370.24  15495.83  15720.70

OBSERVEN: Las primeras filas tienen NaN en lag_1 y lag_7
   Porque no hay datos anteriores disponibles


In [4]:
# CREAR MEDIA MÓVIL SIN FUGA
# shift(1) para excluir el día actual
# rolling(7) para tomar los últimos 7 días
# mean() para promediar

df["media_movil_7"] = (
    df["demanda_kwh"]
    .shift(1)          # EXCLUIR EL DÍA ACTUAL (evita fuga)
    .rolling(window=7, min_periods=7) # Últimos 7 días
    .mean()            # Promedio
)

print("MEDIA MÓVIL DE 7 DÍAS (SIN FUGA):")
print("=" * 80)
print(df[["fecha", "demanda_kwh", "lag_1", "media_movil_7"]].head(12).to_string())

print("\nVERIFICACIÓN DE NO FUGA:")
print("   • media_movil_7 en la fila 10 = promedio de demanda_kwh de filas 3-9")
print("   • NO incluye demanda_kwh de fila 10 (el día actual)")
print("   • Por lo tanto, está disponible antes de predecir")

MEDIA MÓVIL DE 7 DÍAS (SIN FUGA):
        fecha  demanda_kwh     lag_1  media_movil_7
0  2026-01-01     17856.31       NaN            NaN
1  2026-01-02     17379.10  17856.31            NaN
2  2026-01-03     15720.70  17379.10            NaN
3  2026-01-04     15694.05  15720.70            NaN
4  2026-01-05     16505.77  15694.05            NaN
5  2026-01-06     16937.36  16505.77            NaN
6  2026-01-07     17456.69  16937.36            NaN
7  2026-01-08     16568.10  17456.69   16792.854286
8  2026-01-09     15495.83  16568.10   16608.824286
9  2026-01-10     15370.24  15495.83   16339.785714
10 2026-01-11     15652.73  15370.24   16289.720000
11 2026-01-12     16919.53  15652.73   16283.817143

VERIFICACIÓN DE NO FUGA:
   • media_movil_7 en la fila 10 = promedio de demanda_kwh de filas 3-9
   • NO incluye demanda_kwh de fila 10 (el día actual)
   • Por lo tanto, está disponible antes de predecir


In [5]:
# ELIMINAR FILAS CON NaN (PRIMERAS FILAS SIN HISTORIAL)
print(f"Filas ANTES de eliminar NaN: {len(df)}")

features = [
    "dia_semana",
    "mes",
    "evento_programado",
    "lag_1",
    "lag_7",
    "media_movil_7"
]

df_clean = df.dropna(subset=features + ["demanda_kwh"]).copy().reset_index(drop=True)

print(f"Filas DESPUÉS de eliminar NaN: {len(df_clean)}")
print(f"Filas eliminadas: {len(df) - len(df_clean)}")

Filas ANTES de eliminar NaN: 180
Filas DESPUÉS de eliminar NaN: 173
Filas eliminadas: 7


In [6]:
# DEFINIR VARIABLES PREDICTORAS (X) Y OBJETIVO (y)
X = df_clean[features]
y = df_clean["demanda_kwh"]
fechas = df_clean["fecha"]

print("VARIABLES DEFINIDAS:")
print("=" * 60)
print(f"Variables predictoras (X): {len(features)}")
for i, f in enumerate(features, 1):
    print(f"   {i}. {f}")
print(f"\nVariable objetivo (y): demanda_kwh")
print(f"\nDimensiones:")
print(f"   X: {X.shape}")
print(f"   y: {y.shape}")

VARIABLES DEFINIDAS:
Variables predictoras (X): 6
   1. dia_semana
   2. mes
   3. evento_programado
   4. lag_1
   5. lag_7
   6. media_movil_7

Variable objetivo (y): demanda_kwh

Dimensiones:
   X: (173, 6)
   y: (173,)


In [7]:
# SPLIT TEMPORAL 80/20 (CRONOLÓGICO)
corte = int(len(df_clean) * 0.80)

X_train = X.iloc[:corte]
X_test = X.iloc[corte:]

y_train = y.iloc[:corte]
y_test = y.iloc[corte:]

fechas_train = fechas.iloc[:corte]
fechas_test = fechas.iloc[corte:]

print("SPLIT TEMPORAL 80/20")
print("=" * 60)
print(f"Total de registros: {len(df_clean)}")
print(f"Entrenamiento: {len(X_train)} registros (80%)")
print(f"Prueba: {len(X_test)} registros (20%)")

print(f"\nRango de ENTRENAMIENTO:")
print(f"   Desde: {fechas_train.min().strftime('%Y-%m-%d')}")
print(f"   Hasta: {fechas_train.max().strftime('%Y-%m-%d')}")

print(f"\nRango de PRUEBA:")
print(f"   Desde: {fechas_test.min().strftime('%Y-%m-%d')}")
print(f"   Hasta: {fechas_test.max().strftime('%Y-%m-%d')}")

SPLIT TEMPORAL 80/20
Total de registros: 173
Entrenamiento: 138 registros (80%)
Prueba: 35 registros (20%)

Rango de ENTRENAMIENTO:
   Desde: 2026-01-08
   Hasta: 2026-05-25

Rango de PRUEBA:
   Desde: 2026-05-26
   Hasta: 2026-06-29


In [8]:
#Entrenar dummy como baseline y linearRegresion como modelo principal
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)

modelo = LinearRegression()
modelo.fit(X_train, y_train)
pred_modelo = modelo.predict(X_test)

#Metricas (todas sobre el conjunto de PRUEBA)
mae_baseline = mean_absolute_error(y_test, pred_baseline)
mae_modelo   = mean_absolute_error(y_test, pred_modelo)
rmse_modelo  = np.sqrt(mean_squared_error(y_test, pred_modelo))
r2_modelo    = r2_score(y_test, pred_modelo)
mejora_mae_pct = (mae_baseline - mae_modelo) / mae_baseline * 100

print("RESULTADOS EN PRUEBA:")
print(f"Baseline (media del train): {baseline.constant_[0][0]:.2f} kWh")
print(f"MAE baseline : {mae_baseline:.2f}")
print(f"MAE modelo   : {mae_modelo:.2f}")
print(f"RMSE modelo  : {rmse_modelo:.2f}")
print(f"R² modelo    : {r2_modelo:.4f}")
print(f"Mejora MAE   : {mejora_mae_pct:.2f}%")

RESULTADOS EN PRUEBA:
Baseline (media del train): 17281.82 kWh
MAE baseline : 1194.57
MAE modelo   : 298.77
RMSE modelo  : 378.72
R² modelo    : 0.7824
Mejora MAE   : 74.99%


In [9]:
#Exportacion de predicciones.csv (solo conjunto de prueba, orden cronológico)
predicciones = pd.DataFrame({
    "fecha": fechas_test.dt.strftime("%Y-%m-%d").values,
    "real": y_test.values,
    "baseline": pred_baseline,
    "prediccion": pred_modelo
})
ruta_predicciones = DOC_RESULTADOS / 'predicciones.csv'
predicciones.to_csv(ruta_predicciones, index=False)

#Exportacion metricas.json (claves exactas del contrato)
metricas = {
    "mae_baseline": float(mae_baseline),
    "mae_modelo": float(mae_modelo),
    "rmse_modelo": float(rmse_modelo),
    "r2_modelo": float(r2_modelo),
    "mejora_mae_pct": float(mejora_mae_pct)
}
ruta_metrica = DOC_RESULTADOS / 'metricas.json'
with open(ruta_metrica, "w") as f:
    json.dump(metricas, f, indent=4)

#Verificacion
print("predicciones.csv:", predicciones.shape, "(esperado: 35 filas, 4 columnas)")
print(predicciones.head(5).to_string(index=False))
print("\nmetricas.json:")
print(json.dumps(metricas, indent=4))

predicciones.csv: (35, 4) (esperado: 35 filas, 4 columnas)
     fecha     real     baseline   prediccion
2026-05-26 18800.00 17281.821304 18603.319450
2026-05-27 19294.76 17281.821304 18371.300370
2026-05-28 18761.74 17281.821304 18254.034120
2026-05-29 17890.43 17281.821304 17746.713486
2026-05-30 17112.40 17281.821304 17218.995397

metricas.json:
{
    "mae_baseline": 1194.5729937888211,
    "mae_modelo": 298.76500228805673,
    "rmse_modelo": 378.72456527182237,
    "r2_modelo": 0.7823861086344409,
    "mejora_mae_pct": 74.98980775210184
}



## P4
La prueba va del 2026-05-26 al 2026-06-29 (35 días). El MAE del baseline fue 1194.57 kWh y el del modelo 298.77 kWh, una mejora de 74.99 %. No se debe entrenar con datos del futuro porque habría fuga: el modelo vería información que en la vida real no tendría al pronosticar, y las métricas saldrían abultadas, por eso el split es cronológico y media_movil_7 usa shift(1), para no incluir la demanda del día que se predice. El pronóstico solo sirve si se conoce la demanda de ayer y de hace 7 días y el calendario de eventos; además, con solo 173 días de historia el modelo no ve un año completo y no captura estacionalidad anual, así que es una estimación, no una certeza.